# ecf — detection de fake news dans les titres de presse
### pipeline nlp : tf-idf · tensorflow · fastapi

---

## imports

In [ ]:
import os
import re
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import tensorflow as tf
from collections import Counter
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

warnings.filterwarnings("ignore")

# tÃ©lÃ©chargement des ressources nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

# chargement des variables d'environnement
load_dotenv()

print("imports ok")
print(f"tensorflow : {tf.__version__}")

---
## partie 1 — chargement et exploration

### 1.1 chargement du corpus

In [ ]:
def load_titles(filepath: str) -> pd.DataFrame:
    """
    charge le csv et retourne un dataframe avec text et label.
    """
    df = pd.read_csv(filepath)

    # on garde uniquement title et label, on renomme title -> text
    df = df[["title", "label"]].rename(columns={"title": "text"})

    # suppression des lignes vides
    df = df.dropna(subset=["text"])
    df = df[df["text"].str.strip() != ""]
    df = df.reset_index(drop=True)

    # encodage des labels : REAL -> 1, FAKE -> 0
    df["label"] = df["label"].map({"REAL": 1, "FAKE": 0})

    # resume
    print(f"total de titres : {len(df)}")
    print(f"\ndistribution des classes :")
    counts = df["label"].value_counts()
    for label_val, count in counts.items():
        name = "real" if label_val == 1 else "fake"
        print(f"  {name} ({label_val}) : {count} ({count / len(df) * 100:.1f}%)")

    df["nb_tokens"] = df["text"].apply(lambda x: len(x.split()))
    print(f"\nlongueur moyenne des titres : {df['nb_tokens'].mean():.1f} tokens")

    return df


data_path = os.getenv("DATA_PATH", "../data/news.csv")
df = load_titles(data_path)
df.head()

In [ ]:
# sauvegarde du corpus brut nettoyÃ©
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/titles_clean.csv", index=False)
print("fichier sauvegardÃ© : ../data/titles_clean.csv")

### 1.2 analyse exploratoire

In [ ]:
# distribution des classes
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["label"].value_counts()
ax.bar(["fake (0)", "real (1)"], [counts[0], counts[1]], color=["#e74c3c", "#2ecc71"])
ax.set_title("distribution des classes")
ax.set_ylabel("nombre de titres")
for i, v in enumerate([counts[0], counts[1]]):
    ax.text(i, v + 20, str(v), ha="center")
plt.tight_layout()
plt.show()

**distribution des classes**

le corpus est a peu pres equilibre, pas besoin de corriger quoi que ce soit. on utilisera juste `stratify=y` dans le split pour garder la meme proportion dans train et test.

In [ ]:
# distribution de la longueur des titres par classe
df_fake = df[df["label"] == 0]
df_real = df[df["label"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_fake["nb_tokens"], bins=30, color="#e74c3c", edgecolor="white")
axes[0].set_title("longueur des titres â€” fake")
axes[0].set_xlabel("nombre de tokens")
axes[0].set_ylabel("frÃ©quence")

axes[1].hist(df_real["nb_tokens"], bins=30, color="#2ecc71", edgecolor="white")
axes[1].set_title("longueur des titres â€” real")
axes[1].set_xlabel("nombre de tokens")
axes[1].set_ylabel("frÃ©quence")

plt.tight_layout()
plt.show()

for name, subset in [("fake", df_fake), ("real", df_real)]:
    print(f"{name} â€” min: {subset['nb_tokens'].min()}, max: {subset['nb_tokens'].max()}, mÃ©diane: {subset['nb_tokens'].median():.0f}")

**longueur des titres**

les titres fake ont tendance a etre un peu plus longs. les titres real sont plus courts et plus directs. on utilise la mediane plutot que la moyenne car elle n'est pas perturbee par les titres extremement longs ou courts.

In [ ]:
# top 20 tokens les plus frÃ©quents par classe
def get_top_tokens(texts, n=20):
    all_words = []
    for text in texts:
        all_words.extend(str(text).lower().split())
    return Counter(all_words).most_common(n)


top_fake = get_top_tokens(df_fake["text"])
top_real = get_top_tokens(df_real["text"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

words_f, counts_f = zip(*top_fake)
axes[0].barh(words_f[::-1], counts_f[::-1], color="#e74c3c")
axes[0].set_title("top 20 tokens â€” fake")
axes[0].set_xlabel("frÃ©quence")

words_r, counts_r = zip(*top_real)
axes[1].barh(words_r[::-1], counts_r[::-1], color="#2ecc71")
axes[1].set_title("top 20 tokens â€” real")
axes[1].set_xlabel("frÃ©quence")

plt.tight_layout()
plt.show()

**top 20 tokens**

sans surprise, les mots les plus frequents dans les deux classes sont des stopwords (the, to, a...). ca ne sert a rien pour differencier fake de real. c'est pour ca qu'on va les supprimer dans la partie 2 pour faire ressortir ce qui compte vraiment.

In [ ]:
# tokens discriminants purs (prÃ©sents dans une seule classe)
vocab_fake = set()
for text in df_fake["text"]:
    vocab_fake.update(str(text).lower().split())

vocab_real = set()
for text in df_real["text"]:
    vocab_real.update(str(text).lower().split())

only_fake = vocab_fake - vocab_real
only_real = vocab_real - vocab_fake

print(f"tokens prÃ©sents uniquement dans fake : {len(only_fake)}")
print(f"10 premiers : {list(only_fake)[:10]}")
print()
print(f"tokens prÃ©sents uniquement dans real : {len(only_real)}")
print(f"10 premiers : {list(only_real)[:10]}")

**tokens discriminants**

ces tokens n'apparaissent que dans une seule classe, ils peuvent etre de bons indicateurs pour le modele. attention quand meme aux mots tres rares qui risquent de sur-apprendre. le parametre `min_df=2` du tfidf s'en occupe : il ignore tout token qui apparait dans moins de 2 documents.

In [ ]:
# 3 titres potentiellement ambigus
print("exemples de titres potentiellement ambigus :")
print()

# titres qui mÃ©langent un ton factuel avec un sujet sensible
ambiguous_indices = []
for i, row in df.iterrows():
    title = str(row["text"])
    # titres avec des mots potentiellement neutres mais sujet controversÃ©
    if any(word in title.lower() for word in ["report", "says", "claims", "sources"]):
        ambiguous_indices.append(i)
    if len(ambiguous_indices) >= 3:
        break

for idx in ambiguous_indices[:3]:
    row = df.loc[idx]
    label_name = "real" if row["label"] == 1 else "fake"
    print(f"  titre : {row['text']}")
    print(f"  classe rÃ©elle : {label_name}")
    print()

**titres ambigus**

des mots comme "says", "claims" ou "sources" peuvent apparaitre dans des articles serieux mais aussi dans des fake news qui inventent de fausses citations. le modele devra s'appuyer sur tout le titre, pas un seul mot. les bigrammes dans le tfidf vont aider pour ca.

---
## partie 2 — nettoyage et pretraitement

### 2.1 pipeline de nettoyage

In [ ]:
from nltk.corpus import stopwords

# chargement du modÃ¨le spacy
nlp = spacy.load("en_core_web_sm")

# stopwords nltk â€” on conserve les mots de nÃ©gation
negation_words = {"not", "no", "never", "neither"}
stop_words = set(stopwords.words("english")) - negation_words

# dictionnaire de contractions anglaises (20+)
contractions = {
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "won't": "will not",
    "wouldn't": "would not",
    "can't": "cannot",
    "couldn't": "could not",
    "shouldn't": "should not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "they're": "they are",
    "we're": "we are",
    "you're": "you are",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "i'm": "i am",
    "i've": "i have",
    "i'll": "i will",
    "i'd": "i would",
    "they've": "they have",
    "we've": "we have",
    "you've": "you have",
    "he's": "he is",
    "she's": "she is",
}


def expand_contractions(text: str) -> str:
    """remplace les contractions par leur forme complÃ¨te."""
    for contraction, expanded in contractions.items():
        text = text.replace(contraction, expanded)
    return text


def clean_title(text: str) -> str:
    """
    pipeline de nettoyage d'un titre.
    Ã©tapes : minuscules â†’ urls/mentions â†’ ponctuation/chiffres â†’
             contractions â†’ stopwords â†’ lemmatisation â†’ tokens courts
    """
    # 1. mise en minuscules
    text = text.lower()

    # 2. suppression des urls et mentions @username
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)

    # 3. suppression de la ponctuation et des chiffres isolÃ©s
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)

    # 4. expansion des contractions
    text = expand_contractions(text)

    # 5. suppression des stopwords (sauf mots de nÃ©gation)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]

    # 6. lemmatisation avec spacy
    doc = nlp(" ".join(tokens))
    lemmas = [token.lemma_ for token in doc]

    # 7. suppression des tokens de longueur < 2
    lemmas = [t for t in lemmas if len(t) >= 2]

    return " ".join(lemmas)


# test sur un exemple
example = "Scientists don't want you to know this SECRET about vaccines!"
print(f"original : {example}")
print(f"nettoyÃ©  : {clean_title(example)}")

In [ ]:
# application sur tout le corpus
print("nettoyage en cours...")
df["text_clean"] = df["text"].apply(clean_title)
print("nettoyage terminÃ©.")
df[["text", "text_clean", "label"]].head(5)

### 2.2 impact du nettoyage

In [ ]:
# vocabulaire avant nettoyage
vocab_before = set()
for text in df["text"]:
    vocab_before.update(str(text).lower().split())

# vocabulaire aprÃ¨s nettoyage
vocab_after = set()
for text in df["text_clean"]:
    vocab_after.update(str(text).split())

# longueurs avant/aprÃ¨s
df["len_before"] = df["text"].apply(lambda x: len(str(x).split()))
df["len_after"] = df["text_clean"].apply(lambda x: len(str(x).split()))

reduction = (df["len_before"] - df["len_after"]).mean()

print(f"taille du vocabulaire avant nettoyage : {len(vocab_before)}")
print(f"taille du vocabulaire aprÃ¨s nettoyage : {len(vocab_after)}")
print(f"rÃ©duction : {len(vocab_before) - len(vocab_after)} tokens ({(1 - len(vocab_after)/len(vocab_before))*100:.1f}%)")
print(f"\nrÃ©duction moyenne de longueur par titre : {reduction:.1f} tokens")

# titres vides aprÃ¨s nettoyage
empty_after = (df["text_clean"].str.strip() == "").sum()
print(f"\ntitres devenus vides aprÃ¨s nettoyage : {empty_after}")

In [ ]:
# gestion des titres vides
if empty_after > 0:
    df = df[df["text_clean"].str.strip() != ""].reset_index(drop=True)
    print(f"titres vides supprimÃ©s. corpus final : {len(df)} titres")
else:
    print("aucun titre vide, rien Ã  supprimer.")

**question — pourquoi garder les mots de negation ?**

si on supprime "not", "no", "never", "neither" avec les autres stopwords, on inverse le sens de la phrase.

deux exemples concrets :
1. *"government does not hide vaccine data"* (real) -> sans "not" ca devient *"government hide vaccine data"* et le modele risque de le classer fake.
2. *"scientists say there is no link between autism and vaccines"* (real) -> sans "no" ca donne *"scientists say link autism vaccines"* qui ressemble a du fake.

garder ces mots est essentiel pour ne pas retourner le sens des titres.

---
## partie 3 — representation vectorielle

### 3.1 vectorisation tf-idf

In [ ]:
# split train/test â€” 80/20, stratifiÃ©, reproductible
X = df["text_clean"].values
y = df["label"].values

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train : {len(X_train_text)} exemples")
print(f"test  : {len(X_test_text)} exemples")

# conserver aussi les titres bruts pour l'analyse des erreurs
X_raw = df["text"].values
X_train_raw, X_test_raw, _, _ = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# vectorisation tf-idf
vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

# fit uniquement sur le train, transform train et test
X_train_tfidf = vectorizer.fit_transform(X_train_text).toarray()
X_test_tfidf = vectorizer.transform(X_test_text).toarray()

print(f"shape train tfidf : {X_train_tfidf.shape}")
print(f"shape test tfidf  : {X_test_tfidf.shape}")

# sauvegarde du vectoriseur
os.makedirs("../models", exist_ok=True)
joblib.dump(vectorizer, "../models/vectorizer.pkl")
print("vectoriseur sauvegardÃ© : ../models/vectorizer.pkl")

### 3.2 embeddings avec tensorflow

In [ ]:
# couche textvectorization pour les embeddings appris
# on utilise les titres bruts (non lemmatisÃ©s) pour cette reprÃ©sentation
X_raw = df["text"].values
X_train_raw_embed, X_test_raw_embed, _, _ = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

text_vectorization = layers.TextVectorization(
    max_tokens=5000,
    output_sequence_length=30,
)

# adaptation uniquement sur les donnÃ©es d'entraÃ®nement
text_vectorization.adapt(X_train_raw_embed)

print(f"taille du vocabulaire appris : {len(text_vectorization.get_vocabulary())}")
print("couche textvectorization prÃªte.")

**question — tf-idf vs embedding appris**

| | tf-idf | embedding appris |
|---|---|---|
| type de vecteur | creux (beaucoup de zeros) | dense (tous les chiffres ont une valeur) |
| semantique | non — chaque mot est independant | oui — mots proches ont des vecteurs proches |
| taille | 3000 dimensions fixes | 64 dimensions apprises |

seul l'embedding peut comprendre que `misleading` et `deceptive` veulent dire la meme chose, parce qu'il apprend les representations en regardant dans quel contexte les mots apparaissent. en tf-idf, ces deux mots ont des colonnes separees et aucun lien entre elles.

---
## partie 4 — modelisation

### 4.1 modele dense sur tf-idf (baseline)

In [ ]:
def build_dense_model(input_dim: int) -> keras.Sequential:
    """construit le modÃ¨le dense sur vecteurs tf-idf."""
    model = keras.Sequential(
        [
            layers.Input(shape=(input_dim,)),
            layers.Dense(256, activation="relu"),
            layers.Dropout(0.4),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.3),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="dense_tfidf",
    )
    return model


model_dense = build_dense_model(input_dim=X_train_tfidf.shape[1])
model_dense.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model_dense.summary()

In [ ]:
os.makedirs("../models", exist_ok=True)

callbacks_dense = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
    ModelCheckpoint(
        filepath="../models/best_model.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
]

start_dense = time.time()
history_dense = model_dense.fit(
    X_train_tfidf,
    y_train,
    epochs=30,
    validation_split=0.15,
    callbacks=callbacks_dense,
    verbose=1,
)
time_dense = time.time() - start_dense
print(f"entraÃ®nement terminÃ© en {time_dense:.1f}s")

In [ ]:
# courbes d'apprentissage â€” modÃ¨le dense
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_dense.history["loss"], label="train")
axes[0].plot(history_dense.history["val_loss"], label="validation")
axes[0].set_title("loss â€” modÃ¨le dense (tf-idf)")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()

axes[1].plot(history_dense.history["accuracy"], label="train")
axes[1].plot(history_dense.history["val_accuracy"], label="validation")
axes[1].set_title("accuracy â€” modÃ¨le dense (tf-idf)")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

epochs_dense = len(history_dense.history["loss"])
print(f"epochs effectifs : {epochs_dense}")

### 4.2 modele bilstm avec embeddings

In [ ]:
def build_lstm_model(text_vec_layer) -> keras.Sequential:
    """construit le modÃ¨le bilstm avec embeddings appris."""
    model = keras.Sequential(
        [
            text_vec_layer,
            layers.Embedding(input_dim=5000, output_dim=64, mask_zero=True),
            layers.Bidirectional(
                layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2)
            ),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.3),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="bilstm_embedding",
    )
    return model


model_lstm = build_lstm_model(text_vectorization)
model_lstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model_lstm.summary()

In [ ]:
callbacks_lstm = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
    ModelCheckpoint(
        filepath="../models/best_model_lstm.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
]

start_lstm = time.time()
history_lstm = model_lstm.fit(
    X_train_raw_embed,
    y_train,
    epochs=30,
    validation_split=0.15,
    callbacks=callbacks_lstm,
    verbose=1,
)
time_lstm = time.time() - start_lstm
print(f"entraÃ®nement terminÃ© en {time_lstm:.1f}s")

In [ ]:
# courbes d'apprentissage â€” modÃ¨le lstm
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_lstm.history["loss"], label="train")
axes[0].plot(history_lstm.history["val_loss"], label="validation")
axes[0].set_title("loss â€” bilstm (embeddings)")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend()

axes[1].plot(history_lstm.history["accuracy"], label="train")
axes[1].plot(history_lstm.history["val_accuracy"], label="validation")
axes[1].set_title("accuracy â€” bilstm (embeddings)")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

epochs_lstm = len(history_lstm.history["loss"])
print(f"epochs effectifs : {epochs_lstm}")

### 4.3 comparaison des deux modeles

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Ã©valuation modÃ¨le dense
y_pred_dense_proba = model_dense.predict(X_test_tfidf).flatten()
y_pred_dense = (y_pred_dense_proba >= 0.5).astype(int)

acc_dense = (y_pred_dense == y_test).mean()
prec_dense = precision_score(y_test, y_pred_dense, pos_label=0)
rec_dense = recall_score(y_test, y_pred_dense, pos_label=0)
f1_dense = f1_score(y_test, y_pred_dense, average="macro")
auc_dense = roc_auc_score(y_test, y_pred_dense_proba)

# Ã©valuation modÃ¨le lstm
y_pred_lstm_proba = model_lstm.predict(X_test_raw_embed).flatten()
y_pred_lstm = (y_pred_lstm_proba >= 0.5).astype(int)

acc_lstm = (y_pred_lstm == y_test).mean()
prec_lstm = precision_score(y_test, y_pred_lstm, pos_label=0)
rec_lstm = recall_score(y_test, y_pred_lstm, pos_label=0)
f1_lstm = f1_score(y_test, y_pred_lstm, average="macro")
auc_lstm = roc_auc_score(y_test, y_pred_lstm_proba)

params_dense = model_dense.count_params()
params_lstm = model_lstm.count_params()

# tableau comparatif
comparison = pd.DataFrame(
    {
        "critÃ¨re": [
            "accuracy (test)",
            "precision â€” classe fake",
            "recall â€” classe fake",
            "f1-score (macro)",
            "auc-roc",
            "epochs effectifs",
            "nb paramÃ¨tres entraÃ®nables",
            "temps entraÃ®nement (s)",
        ],
        "modÃ¨le dense (tf-idf)": [
            f"{acc_dense:.4f}",
            f"{prec_dense:.4f}",
            f"{rec_dense:.4f}",
            f"{f1_dense:.4f}",
            f"{auc_dense:.4f}",
            epochs_dense,
            params_dense,
            f"{time_dense:.1f}",
        ],
        "modÃ¨le bilstm": [
            f"{acc_lstm:.4f}",
            f"{prec_lstm:.4f}",
            f"{rec_lstm:.4f}",
            f"{f1_lstm:.4f}",
            f"{auc_lstm:.4f}",
            epochs_lstm,
            params_lstm,
            f"{time_lstm:.1f}",
        ],
    }
)

print(comparison.to_string(index=False))

**question — quel modele choisir en production ?**

je recommande le **modele dense (tf-idf)** pour ces raisons :

1. **rapidite** : la vectorisation tfidf est quasi-instantanee. le bilstm traite les sequences mot par mot, c'est plus lent.
2. **simplicite** : tfidf + dense = facile a maintenir et a debugger. le bilstm est plus complexe.
3. **titres courts** : avec des titres de moins de 15 mots en moyenne, le bilstm n'apporte pas grand chose de plus.
4. **deploiement** : vectorizer pkl + modele dense = tres leger. plus simple a mettre en prod.

si le bilstm etait meilleur de plus de 2-3% d'accuracy, ca vaudrait la peine de changer d'avis.

---
## partie 5 — evaluation approfondie

### 5.1 performances du meilleur modele

In [ ]:
# on Ã©value le meilleur modÃ¨le (dense tf-idf, sauvegardÃ© par modelcheckpoint)
best_model = keras.models.load_model("../models/best_model.keras")
y_pred_proba = best_model.predict(X_test_tfidf).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

# matrice de confusion
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm / cm.sum() * 100

fig, ax = plt.subplots(figsize=(6, 5))
labels = [[f"{v}\n({p:.1f}%)" for v, p in zip(row_v, row_p)] for row_v, row_p in zip(cm, cm_pct)]
sns.heatmap(
    cm,
    annot=labels,
    fmt="",
    cmap="Blues",
    xticklabels=["fake", "real"],
    yticklabels=["fake", "real"],
    ax=ax,
)
ax.set_title("matrice de confusion â€” meilleur modÃ¨le")
ax.set_xlabel("prÃ©dit")
ax.set_ylabel("rÃ©el")
plt.tight_layout()
plt.show()

In [ ]:
# rapport de classification
print("rapport de classification complet :")
print(classification_report(y_test, y_pred, target_names=["fake", "real"]))

In [ ]:
# courbe roc
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#3498db", label=f"auc = {auc:.4f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="alÃ©atoire")
ax.set_title("courbe roc â€” meilleur modÃ¨le")
ax.set_xlabel("taux faux positifs")
ax.set_ylabel("taux vrais positifs")
ax.legend()
plt.tight_layout()
plt.show()

### 5.2 analyse des erreurs

In [ ]:
# construction d'un dataframe d'analyse
df_test = pd.DataFrame(
    {
        "title_raw": X_test_raw,
        "label_true": y_test,
        "label_pred": y_pred,
        "score": y_pred_proba,
    }
)

# faux positifs : REAL classifiÃ©s FAKE (label_true=1, label_pred=0)
faux_positifs = (
    df_test[(df_test["label_true"] == 1) & (df_test["label_pred"] == 0)]
    .assign(confidence=lambda d: 1 - d["score"])  # confiance dans la mauvaise classe
    .sort_values("confidence", ascending=False)
    .head(15)
)

print("15 faux positifs (real classifiÃ©s fake) â€” score de confiance le plus Ã©levÃ© :")
for _, row in faux_positifs.iterrows():
    print(f"  [{row['confidence']:.3f}] {row['title_raw']}")

In [ ]:
# faux nÃ©gatifs : FAKE classifiÃ©s REAL (label_true=0, label_pred=1)
faux_negatifs = (
    df_test[(df_test["label_true"] == 0) & (df_test["label_pred"] == 1)]
    .assign(confidence=lambda d: d["score"])  # confiance dans la mauvaise classe
    .sort_values("confidence", ascending=False)
    .head(15)
)

print("15 faux nÃ©gatifs (fake classifiÃ©s real) â€” score de confiance le plus Ã©levÃ© :")
for _, row in faux_negatifs.iterrows():
    print(f"  [{row['confidence']:.3f}] {row['title_raw']}")

**commentaire — erreurs du modele**

**faux positifs (real classe fake) :** le modele se trompe surtout sur des titres politiques avec des mots forts ou emotionnels. le style ressemble aux fake news meme si le contenu est vrai.

**faux negatifs (fake classe real) :** les fake les plus durs a detecter sont ceux qui copient un style journalistique neutre, sans aucun mot sensationnaliste. le modele ne voit pas la difference avec un vrai article.

### 5.3 test de robustesse — 10 nouveaux titres

In [ ]:
new_titles = [
    "scientists discover new treatment for common disease",
    "shocking: government hiding truth about water supply",
    "local elections results announced in three counties",
    "you won't believe what this celebrity did last night",
    "central bank raises interest rates by 0.25 points",
    "this one weird trick cures all allergies naturally",
    "parliament votes on new environmental legislation",
    "doctors don't want you to know this secret remedy",
    "tech company reports quarterly earnings below forecast",
    "exclusive: famous actor reveals hidden agenda of elites",
]

# nettoyage et vectorisation
new_titles_clean = [clean_title(t) for t in new_titles]
new_titles_tfidf = vectorizer.transform(new_titles_clean).toarray()

# prÃ©diction
scores = best_model.predict(new_titles_tfidf).flatten()

print(f"{'titre':<65} {'classe':>6} {'confiance':>10}")
print("-" * 85)
for title, score in zip(new_titles, scores):
    label = "real" if score >= 0.5 else "fake"
    confidence = score if score >= 0.5 else 1 - score
    short_title = title[:62] + "..." if len(title) > 62 else title
    print(f"{short_title:<65} {label:>6} {confidence:>10.3f}")

**commentaire — robustesse**

les predictions sont globalement coherentes. les titres avec "shocking", "secret", "won't believe", "exclusive" sont bien detectes comme fake. les titres neutres et factuels sont bien identifies comme real.

le modele peut hesiter sur *"scientists discover new treatment for common disease"* car ce style tres generique est aussi utilise dans les fake news sante. c'est normal : le modele ne voit que le titre, pas le corps de l'article.